**STEP 1: Simulation**

In [1]:
import numpy as np
from scipy.stats import norm
def simulate_matrix(n, d, density, out_fra, corr):
    # Generate a simulated large-scale matrix ~ N(0,1)
    X = np.random.normal(loc=0, scale=1, size=(n, d))

    # Apply corr structure with normal distribution
                                       
    # Set the correlation strength to be the desired value
    corr_strength = corr
    # scaling the values in X with strength
    XX = X * corr_strength
    # Add noise to the correlation values
    XX = XX + np.random.normal(0,1-corr_strength**2,size=(n,d))
    # transform X to be correlated
    X = norm.cdf(XX)

    # Apply nonlinear by using ReLU Function
    X = np.maximum(0,X)

    # Add outliers by randomly selecting rows and adding 10 times entities ~ N(0,1)

    ### size of outliers
    size = int(out_fra * n)
    outlier_idx = np.random.choice(n, size=size, replace=False)
    X[outlier_idx] += 39 * np.random.normal(loc=0, scale=1, size=(size, d))

    # To sparse matrix by setting the desired fraction to be zero (out of density)
    X[np.random.rand(n, d) > density] = 0

    return X

# Matrix Settled
n = 5000       # number of rows         
d = 300        # number of columns
density = 0.87  # sparse density to make more 0 entities
out_fra = 0.2  # outliers' fraction
corr = 0.15    # correlation strength

X = simulate_matrix(n, d, density, out_fra, corr) # A matrix
Y = np.random.normal(loc=0,scale=1,size=(n,1)) # b vector
print("The simulated matrix is:")
print(np.array(X))
print("The number of 0 entities' ratio is:",int((X==0).sum())/(n*d))

The simulated matrix is:
[[ 5.77243441e-01  9.91564098e-01  4.16745352e-01 ...  2.70381412e-01
   5.71888061e-03  2.23352899e-01]
 [ 1.36300451e-01  0.00000000e+00  2.10728073e-01 ...  4.76388259e-01
   1.41043282e-02  1.16082152e-01]
 [ 1.53249926e-01  1.01590681e-01  9.64103056e-01 ...  4.56230661e-01
   5.63225548e-01  0.00000000e+00]
 ...
 [ 5.62258221e-01  6.73501143e-01  3.51415945e-01 ...  0.00000000e+00
   0.00000000e+00  4.54135123e-01]
 [ 0.00000000e+00  7.44149077e-01  9.03376530e-01 ...  8.74858236e-01
   2.99817067e-01  4.45480412e-01]
 [-7.40636294e+01  7.75392391e+01 -1.86090801e+01 ... -4.25068209e+01
   4.79872152e+01  5.33576613e+01]]
The number of 0 entities' ratio is: 0.13014133333333333


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import lsqr
from scipy.linalg import cho_factor, cho_solve
def solver(A,b,solver):
    if solver == 'lsqr':
    # lsqr algorithm
        X = csr_matrix(A)
        Y = np.array(b)
        x_star = lsqr(X, Y)[0]
    # cholesky decomposition algorithm
    elif solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b

    return x_star
x_star_lsqr = solver(X,Y,'lsqr')
x_star_cholesky = solver(X,Y,'cholesky')
x_star_direct = solver(X,Y,'direct')
norm_lsqr = calculate_the_norm_square(X,Y,x_star_lsqr)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:")
print(norm_lsqr, norm_cholesky, norm_direct)
minimum_norm_square = min(norm_lsqr, norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)

The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:
25733231.571434066 4539.384176444465 4539.384176444465
Here we output the minimum of the norm suqare for x_star calculated as:
4539.384176444465


**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [ ]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar


_IncompleteInputError: incomplete input (1783957440.py, line 2)